# Prompt 15: Aggregations and Grouping
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## groupBy() and agg() — Core Pattern

```python
df.groupBy('col1', 'col2')           # returns GroupedData
  .agg(
      F.sum('amount').alias('total'),
      F.avg('amount').alias('avg_amount'),
      F.count('*').alias('n'),
      F.max('amount').alias('max_amt'),
      F.min('amount').alias('min_amt'),
  )
```

Key rule: `groupBy()` returns a **GroupedData** object. Calling an aggregation method (`.agg()`, `.count()`, `.sum()`, etc.) on it returns a new **DataFrame**.

## Shorthand Aggregation Methods

GroupedData exposes shorthand methods when you need a single aggregation:

```python
df.groupBy('dept').count()           # count of rows per group
df.groupBy('dept').sum('salary')     # sum of salary per group
df.groupBy('dept').avg('salary')     # average per group
df.groupBy('dept').max('salary')     # max per group
df.groupBy('dept').min('salary')     # min per group
df.groupBy('dept').mean('salary')    # alias for avg
```

**But prefer `.agg()` for multiple aggregations** — it is more flexible and avoids multiple passes.

## Aggregate Without groupBy — DataFrame-level Aggregation

```python
# Aggregate the entire DataFrame (no grouping)
df.agg(F.sum('salary'), F.avg('salary'), F.count('*'))
df.select(F.sum('salary'), F.avg('salary'))    # select with aggregate functions
```

## All Aggregate Functions at a Glance

| Function | Description |
|----------|-------------|
| `F.count('col')` | Count non-null values in col |
| `F.count('*')` or `F.count(lit(1))` | Count all rows including nulls |
| `F.countDistinct('col')` | Count unique non-null values |
| `F.sum('col')` | Sum of non-null values |
| `F.sumDistinct('col')` | Sum of distinct values (deprecated in Spark 3.2+) |
| `F.avg('col')` | Average (mean) |
| `F.mean('col')` | Alias for `avg` |
| `F.max('col')` | Maximum |
| `F.min('col')` | Minimum |
| `F.stddev('col')` | Sample standard deviation |
| `F.stddev_pop('col')` | Population standard deviation |
| `F.variance('col')` | Sample variance |
| `F.var_pop('col')` | Population variance |
| `F.first('col')` | First value in group (non-deterministic) |
| `F.last('col')` | Last value in group (non-deterministic) |
| `F.collect_list('col')` | All values as array (with duplicates) |
| `F.collect_set('col')` | Unique values as array (no duplicates) |
| `F.approx_count_distinct('col')` | Approximate distinct count (HyperLogLog) |
| `F.percentile_approx('col', 0.5)` | Approximate median/percentile |

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit

spark = SparkSession.builder \
    .appName('AggregationsGrouping') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

data = [
    (1,  'Alice',   'Engineering', 'senior',   95000.0, 2024, 'Q1'),
    (2,  'Bob',     'Marketing',   'junior',   72000.0, 2024, 'Q1'),
    (3,  'Carol',   'Engineering', 'lead',    110000.0, 2024, 'Q2'),
    (4,  'Dave',    'HR',          'junior',   65000.0, 2024, 'Q1'),
    (5,  'Eve',     'Engineering', 'senior',  105000.0, 2024, 'Q2'),
    (6,  'Frank',   'Marketing',   'senior',   89000.0, 2024, 'Q2'),
    (7,  'Grace',   'HR',          'senior',   78000.0, 2024, 'Q2'),
    (8,  'Henry',   'Engineering', 'junior',   72000.0, 2024, 'Q3'),
    (9,  'Iris',    'Marketing',   'lead',     98000.0, 2024, 'Q3'),
    (10, 'Jack',    'Finance',     'senior',   91000.0, 2024, 'Q3'),
    (11, 'Karen',   'Finance',     'junior',   67000.0, 2024, 'Q4'),
    (12, 'Leo',     'Engineering', 'lead',    115000.0, 2024, 'Q4'),
]
schema = 'id INT, name STRING, department STRING, level STRING, salary DOUBLE, year INT, quarter STRING'
df = spark.createDataFrame(data, schema)
df.show()

In [ ]:
# Cell 2: groupBy + agg — single and multiple aggregations

print('=== Single aggregation — shorthand methods ===')
df.groupBy('department').count().show()
df.groupBy('department').sum('salary').show()
df.groupBy('department').avg('salary').show()
df.groupBy('department').max('salary').show()
df.groupBy('department').min('salary').show()

print('=== Multiple aggregations with .agg() ===')
df.groupBy('department').agg(
    F.count('*').alias('headcount'),
    F.sum('salary').alias('total_salary'),
    F.avg('salary').alias('avg_salary'),
    F.max('salary').alias('max_salary'),
    F.min('salary').alias('min_salary'),
    F.stddev('salary').alias('stddev_salary'),
).orderBy('department').show()

print('=== Group by multiple columns ===')
df.groupBy('department', 'level').agg(
    F.count('*').alias('n'),
    F.avg('salary').alias('avg_salary'),
).orderBy('department', 'level').show()

print('=== countDistinct ===')
df.groupBy('department').agg(
    F.countDistinct('level').alias('distinct_levels'),
    F.count('*').alias('total_employees'),
).show()

print('=== DataFrame-level aggregation (no groupBy) ===')
df.agg(
    F.count('*').alias('total_rows'),
    F.sum('salary').alias('company_payroll'),
    F.avg('salary').alias('company_avg_salary'),
    F.max('salary').alias('highest_salary'),
    F.min('salary').alias('lowest_salary'),
).show()

In [ ]:
# Cell 3: collect_list, collect_set, approx_count_distinct, percentile_approx

print('=== collect_list — all values including duplicates ===')
# Collect all employee names per department
df.groupBy('department').agg(
    F.collect_list('name').alias('members'),
    F.collect_list('level').alias('levels_with_dups'),
).show(truncate=False)

print('=== collect_set — unique values only (no duplicates) ===')
df.groupBy('department').agg(
    F.collect_set('level').alias('unique_levels'),
).show(truncate=False)

print('=== Key difference: collect_list vs collect_set ===')
# collect_list preserves order and duplicates; collect_set is unordered and unique
df.groupBy('department').agg(
    F.size(F.collect_list('level')).alias('list_size'),    # same as count
    F.size(F.collect_set('level')).alias('set_size'),      # = distinct count
).show()

print('=== approx_count_distinct — fast approximate distinct count ===')
# Uses HyperLogLog — much faster than countDistinct on large data
df.agg(
    F.countDistinct('level').alias('exact_distinct'),
    F.approx_count_distinct('level').alias('approx_distinct'),
    F.approx_count_distinct('level', 0.05).alias('approx_rsd_5pct'),
).show()

print('=== percentile_approx ===')
df.agg(
    F.percentile_approx('salary', 0.5).alias('median_salary'),
    F.percentile_approx('salary', 0.25).alias('p25_salary'),
    F.percentile_approx('salary', 0.75).alias('p75_salary'),
    F.percentile_approx('salary', [0.1, 0.5, 0.9]).alias('deciles'),
).show(truncate=False)

print('=== first() and last() — non-deterministic without sort ===')
df.groupBy('department').agg(
    F.first('name').alias('first_name'),
    F.last('name').alias('last_name'),
    F.first('name', ignorenulls=True).alias('first_non_null_name'),
).show()

## pivot() — Reshape Rows Into Columns

`pivot()` transforms unique values of a column into new columns, then aggregates.

```python
# Pattern:
df.groupBy('row_key') \
  .pivot('pivot_col')              # each unique value becomes a column
  .agg(F.sum('value_col'))         # aggregate into each cell
```

**Performance tip:** Always provide the pivot values explicitly to avoid a full scan:
```python
# Without values — Spark scans all data first to discover unique values (2 passes)
df.groupBy('dept').pivot('quarter').sum('salary')

# With values — single pass, much faster on large datasets
df.groupBy('dept').pivot('quarter', ['Q1', 'Q2', 'Q3', 'Q4']).sum('salary')
```

**NULL in pivot cells:** If no data exists for a combination, the cell is `NULL`.
Use `F.coalesce()` or `fillna()` to substitute 0.

## rollup() — Hierarchical Subtotals

`rollup()` creates subtotals at each level of a column hierarchy, plus a grand total.

```python
df.rollup('department', 'level').agg(F.sum('salary').alias('total'))
```

Produces groups: `(dept, level)`, `(dept, NULL)` = dept subtotal, `(NULL, NULL)` = grand total.
The `NULL` in rollup output means **"all values"** — it is NOT a data NULL.

## cube() — All Combinations of Subtotals

`cube()` generates aggregations for **all combinations** of the specified columns.

```python
df.cube('department', 'level').agg(F.sum('salary').alias('total'))
```

Produces: `(dept, level)`, `(dept, NULL)`, `(NULL, level)`, `(NULL, NULL)` = grand total.

**rollup vs cube comparison:**

| Groups generated | rollup(A, B) | cube(A, B) |
|-----------------|-------------|------------|
| (A, B) | ✅ | ✅ |
| (A, NULL) — A subtotal | ✅ | ✅ |
| (NULL, B) — B subtotal | ❌ | ✅ |
| (NULL, NULL) — grand total | ✅ | ✅ |

Use `grouping()` function to distinguish rollup/cube NULLs from data NULLs:
```python
from pyspark.sql.functions import grouping
df.rollup('dept', 'level') \
  .agg(F.sum('salary'), grouping('dept').alias('is_dept_subtotal'))
```

In [ ]:
# Cell 4: pivot() — rows to columns

print('=== pivot(): total salary per department per quarter ===')
pivot_df = df.groupBy('department') \
             .pivot('quarter', ['Q1', 'Q2', 'Q3', 'Q4']) \
             .agg(F.sum('salary'))
pivot_df.show()

print('=== pivot() without specifying values (auto-discovers) ===')
# Auto-discovery requires 2 scans — slower
df.groupBy('department').pivot('quarter').sum('salary').show()

print('=== pivot() headcount ===')
df.groupBy('department') \
  .pivot('quarter', ['Q1', 'Q2', 'Q3', 'Q4']) \
  .count() \
  .fillna(0) \
  .show()

print('=== pivot() avg salary per level per department ===')
df.groupBy('level') \
  .pivot('department', ['Engineering', 'Finance', 'HR', 'Marketing']) \
  .agg(F.round(F.avg('salary'), 0).alias('avg_sal')) \
  .show()

print('=== Handling NULLs in pivot cells with fillna ===')
pivot_df.fillna(0).show()

print('=== Multiple aggs in pivot (Spark generates col_aggfunc naming) ===')
df.groupBy('department') \
  .pivot('quarter', ['Q1', 'Q2']) \
  .agg(
      F.sum('salary').alias('total'),
      F.count('*').alias('n'),
  ) \
  .show()

In [ ]:
# Cell 5: rollup() and cube() — hierarchical aggregations

print('=== rollup(): dept -> level hierarchy with subtotals ===')
rollup_df = df.rollup('department', 'level') \
              .agg(F.sum('salary').alias('total_salary'),
                   F.count('*').alias('n')) \
              .orderBy('department', 'level')
rollup_df.show(30)

print('=== Understanding rollup NULLs ===')
# NULL in dept = grand total; NULL in level = dept subtotal
print('Rows where level IS NULL (dept subtotals + grand total):')
rollup_df.filter(col('level').isNull()).show()

print('=== cube(): all combinations ===')
cube_df = df.cube('department', 'level') \
            .agg(F.sum('salary').alias('total_salary'),
                 F.count('*').alias('n')) \
            .orderBy('department', 'level')
cube_df.show(40)

print('=== cube() generates more rows than rollup() ===')
print(f'rollup rows: {rollup_df.count()}')
print(f'cube rows:   {cube_df.count()}')

print('=== Using grouping() to identify subtotal rows ===')
from pyspark.sql.functions import grouping

df.rollup('department', 'level') \
  .agg(
      F.sum('salary').alias('total'),
      grouping('department').alias('dept_is_subtotal'),  # 1 = this is a subtotal row
      grouping('level').alias('level_is_subtotal'),
  ) \
  .orderBy('department', 'level') \
  .show(30)

In [ ]:
# Cell 6: Conditional aggregation and filtering aggregated results

print('=== Conditional aggregation with when() inside agg ===')
# Count how many senior vs non-senior per department
df.groupBy('department').agg(
    F.count('*').alias('total'),
    F.sum(F.when(col('level') == 'senior', 1).otherwise(0)).alias('senior_count'),
    F.sum(F.when(col('level') == 'junior', 1).otherwise(0)).alias('junior_count'),
    F.sum(F.when(col('level') == 'lead', 1).otherwise(0)).alias('lead_count'),
).show()

print('=== Aggregating conditionally — sum only where condition is true ===')
df.groupBy('department').agg(
    F.sum('salary').alias('total_salary'),
    F.sum(F.when(col('level') == 'senior', col('salary'))).alias('senior_payroll'),
    F.avg(F.when(col('level') != 'junior', col('salary'))).alias('avg_excl_junior'),
).show()

print('=== Filtering after aggregation with .filter() on agg result ===')
# Find departments where average salary > 85000
df.groupBy('department') \
  .agg(F.avg('salary').alias('avg_salary')) \
  .filter(col('avg_salary') > 85000) \
  .orderBy(col('avg_salary').desc()) \
  .show()

print('=== HAVING clause equivalent (SQL) ===')
df.createOrReplaceTempView('employees')
spark.sql("""
    SELECT department,
           COUNT(*)        AS headcount,
           AVG(salary)     AS avg_salary,
           SUM(salary)     AS total_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 85000
    ORDER BY avg_salary DESC
""").show()

print('=== Named agg columns — rename with alias ===')
result = df.groupBy('department', 'quarter').agg(
    F.count('*').alias('headcount'),
    F.sum('salary').alias('total'),
    F.round(F.avg('salary'), 2).alias('avg'),
)
print(f'Result schema: {result.columns}')
result.show()

In [ ]:
# Cell 7: Full realistic ETL — quarterly sales report with pivot + rollup

# Build quarterly headcount and payroll summary
print('=== Step 1: Department quarterly summary ===')
quarterly = df.groupBy('department', 'quarter').agg(
    F.count('*').alias('headcount'),
    F.sum('salary').alias('total_salary'),
    F.avg('salary').alias('avg_salary'),
)
quarterly.show()

print('=== Step 2: Pivot quarterly headcount — departments as rows, quarters as columns ===')
pivot_hc = df.groupBy('department') \
             .pivot('quarter', ['Q1', 'Q2', 'Q3', 'Q4']) \
             .count() \
             .fillna(0)
pivot_hc.show()

print('=== Step 3: Total headcount per department (add column) ===')
pivot_hc_total = pivot_hc.withColumn(
    'annual_total',
    col('Q1') + col('Q2') + col('Q3') + col('Q4')
)
pivot_hc_total.show()

print('=== Step 4: Rollup for executive summary (subtotals at dept and company level) ===')
df.rollup('department', 'level') \
  .agg(
      F.count('*').alias('headcount'),
      F.sum('salary').alias('total_payroll'),
      F.round(F.avg('salary'), 0).alias('avg_salary'),
  ) \
  .fillna({'department': 'GRAND TOTAL', 'level': 'ALL'}) \
  .orderBy('department', 'level') \
  .show(40)

spark.stop()
print('Done.')

## Quick Reference Summary

### groupBy + agg
```python
df.groupBy('col').agg(F.sum('val').alias('s'), F.count('*').alias('n'))
df.groupBy('a', 'b').agg(...)          # multi-column group key
df.agg(F.sum('val'), F.avg('val'))      # whole-DataFrame aggregation
```

### Key aggregate functions
```python
F.count('col')                  # non-null count
F.count('*')                    # all rows (incl. nulls)
F.countDistinct('col')          # exact distinct count
F.approx_count_distinct('col')  # fast approximate distinct
F.sum('col')                    # sum
F.avg('col') / F.mean('col')    # average
F.max('col') / F.min('col')     # extremes
F.stddev('col')                 # sample stddev
F.variance('col')               # sample variance
F.collect_list('col')           # list with duplicates
F.collect_set('col')            # set of unique values
F.first('col', ignorenulls=True)
F.last('col', ignorenulls=True)
F.percentile_approx('col', 0.5) # approximate median
```

### pivot
```python
df.groupBy('row_key') \
  .pivot('col', ['v1', 'v2', 'v3'])   # explicit values = faster (1 pass)
  .agg(F.sum('val'))
.fillna(0)                             # replace pivot NULLs
```

### rollup vs cube
```python
df.rollup('A', 'B').agg(...)  # (A,B), (A,*), (*,*)
df.cube('A', 'B').agg(...)    # (A,B), (A,*), (*,B), (*,*)
# NULL in result = 'all values' subtotal row
# Use grouping('col') to identify which rows are subtotals
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Monthly revenue report with pivot — e-commerce platform</summary>

**Situation:**
A data engineer needs a monthly revenue table where rows are product categories
and columns are months (Jan-Dec), with NULL replaced by 0.

**Code:**
```python
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

monthly_revenue = orders \
    .groupBy('category') \
    .pivot('month', months) \
    .agg(F.round(F.sum('revenue'), 2).alias('rev')) \
    .fillna(0) \
    .orderBy('category')

# Add annual total column
monthly_revenue = monthly_revenue.withColumn(
    'annual_total',
    sum(col(m) for m in months)
)
monthly_revenue.show(truncate=False)
```

**Exam Sub-topic:** `pivot()` with explicit values for performance; `fillna(0)` after pivot; chaining `.withColumn()`
</details>

<details>
<summary>Scenario 2: Hierarchical budget rollup — retail store chain</summary>

**Situation:**
Finance needs a budget vs actual report at store → region → national level.

**Code:**
```python
budget_rollup = sales \
    .rollup('region', 'store_id') \
    .agg(
        F.sum('actual').alias('total_actual'),
        F.sum('budget').alias('total_budget'),
        F.round(
            (F.sum('actual') - F.sum('budget')) / F.sum('budget') * 100, 1
        ).alias('variance_pct'),
        grouping('region').alias('is_rollup'),
    ) \
    .orderBy('region', 'store_id')

# Separate store-level from rollup rows
store_level = budget_rollup.filter(col('is_rollup') == 0)
subtotals   = budget_rollup.filter(col('is_rollup') == 1)
```

**Exam Sub-topic:** `rollup()` multi-level subtotals; `grouping()` to identify rollup rows
</details>

<details>
<summary>Scenario 3: Tag aggregation — collect user tags per session</summary>

**Situation:**
A recommendation engine collects all product tags a user viewed per session
into arrays for downstream collaborative filtering.

**Code:**
```python
user_sessions = events \
    .groupBy('user_id', 'session_id') \
    .agg(
        F.collect_list('product_tag').alias('all_tags'),   # all tags, order preserved
        F.collect_set('product_tag').alias('unique_tags'), # deduped tag set
        F.count('*').alias('event_count'),
        F.min('event_time').alias('session_start'),
        F.max('event_time').alias('session_end'),
    )

# Filter sessions with >= 5 unique tags
rich_sessions = user_sessions.filter(
    F.size(col('unique_tags')) >= 5
)
```

**Exam Sub-topic:** `collect_list` vs `collect_set`; `F.size()` on array column after aggregation
</details>

<details>
<summary>Scenario 4: Statistical summary per cohort — A/B test analysis</summary>

**Situation:**
A/B test results need mean, stddev, and percentile revenue per experiment group
to determine statistical significance.

**Code:**
```python
stats = ab_test_results \
    .groupBy('experiment_group') \
    .agg(
        F.count('*').alias('n'),
        F.avg('revenue').alias('mean_revenue'),
        F.stddev('revenue').alias('stddev_revenue'),
        F.percentile_approx('revenue', 0.5).alias('median_revenue'),
        F.percentile_approx('revenue', [0.25, 0.75]).alias('iqr_bounds'),
        F.sum(F.when(col('converted') == True, 1).otherwise(0)).alias('conversions'),
        F.round(
            F.sum(F.when(col('converted') == True, 1).otherwise(0)) / F.count('*') * 100, 2
        ).alias('conversion_rate_pct'),
    )
stats.show(truncate=False)
```

**Exam Sub-topic:** Multi-function `.agg()`; `stddev()`; `percentile_approx()` with list; conditional sum with `when()`
</details>

<details>
<summary>Scenario 5: HAVING equivalent — filter departments above salary threshold</summary>

**Situation:**
HR wants departments where both average salary > $85k AND at least 3 employees
(SQL HAVING clause equivalent in DataFrame API).

**Code:**
```python
# In DataFrame API: groupBy → agg → filter (filter AFTER agg = HAVING)
eligible_depts = df \
    .groupBy('department') \
    .agg(
        F.count('*').alias('headcount'),
        F.avg('salary').alias('avg_salary'),
    ) \
    .filter(
        (col('avg_salary') > 85000) & (col('headcount') >= 3)
    ) \
    .orderBy(col('avg_salary').desc())

eligible_depts.show()

# Equivalent SQL:
spark.sql("""
    SELECT department, COUNT(*) headcount, AVG(salary) avg_salary
    FROM employees
    GROUP BY department
    HAVING AVG(salary) > 85000 AND COUNT(*) >= 3
    ORDER BY avg_salary DESC
""")
```

**Exam Sub-topic:** `.filter()` after `.agg()` = HAVING; filter references aggregated column aliases
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `groupBy()` return type | `GroupedData` — NOT a DataFrame |
| Calling `.agg()` on GroupedData | Returns a new **DataFrame** |
| `F.count('col')` | Counts **non-null** values |
| `F.count('*')` | Counts ALL rows including nulls |
| `F.countDistinct('col')` | Exact distinct count — triggers shuffle |
| `F.approx_count_distinct('col')` | Approximate — much faster via HyperLogLog |
| `F.avg()` alias | `F.mean()` — identical |
| `F.collect_list('col')` | All values **with duplicates**, order preserved |
| `F.collect_set('col')` | Unique values only — **no order guarantee** |
| `F.first()` / `F.last()` | Non-deterministic without sort |
| `F.percentile_approx('col', 0.5)` | Approximate median |
| `pivot()` with explicit values | **Faster** — single pass vs auto-discover (2 passes) |
| NULL in pivot cells | No data for that combination |
| HAVING equivalent | `.filter()` applied **after** `.agg()` |
| `rollup('A','B')` groups | (A,B), (A,NULL), (NULL,NULL) |
| `cube('A','B')` groups | (A,B), (A,NULL), (NULL,B), (NULL,NULL) |
| NULL in rollup/cube | Means **"all values"** — NOT a data NULL |
| `grouping('col')` | Returns 1 for subtotal rows, 0 for detail rows |
| Conditional aggregation | `F.sum(F.when(condition, col('x')))` |

---

## Practice Q&A

<details>
<summary>Q1: What does groupBy() return and how do you get a DataFrame back?</summary>

**A:** `df.groupBy('col')` returns a **`GroupedData`** object, not a DataFrame.
To get a DataFrame back, call an aggregation method on it:
- `.agg(F.sum('x'), F.count('*'))` — multiple aggregations
- `.count()` — row count per group
- `.sum('x')` — sum per group
- `.avg('x')` / `.mean('x')` — average per group
- `.max('x')` / `.min('x')` — extremes per group
</details>

<details>
<summary>Q2: What is the difference between collect_list() and collect_set()?</summary>

**A:**
- `F.collect_list('col')` — returns an **array of ALL values** in the group, **including duplicates**. Order is preserved within partitions but not guaranteed across partitions.
- `F.collect_set('col')` — returns an array of **unique values only** (deduplication), in no guaranteed order.

Example: if a group has values `['A', 'B', 'A', 'C']`, `collect_list` → `[A, B, A, C]`, `collect_set` → `[A, B, C]` (in any order).
</details>

<details>
<summary>Q3: How do you implement a HAVING clause in the DataFrame API?</summary>

**A:** Apply `.filter()` **after** `.agg()`. Spark pushes this filter down after the aggregation, just like SQL HAVING:

```python
df.groupBy('dept') \
  .agg(F.avg('salary').alias('avg_sal'), F.count('*').alias('n')) \
  .filter((col('avg_sal') > 80000) & (col('n') >= 3))
```

**Key distinction:** `.filter()` BEFORE `groupBy` = SQL WHERE (filters rows). `.filter()` AFTER `.agg()` = SQL HAVING (filters groups).
</details>

<details>
<summary>Q4: What is the difference between rollup() and cube()?</summary>

**A:** Both add subtotal rows with `NULL` representing "all values", but they differ in which combinations are included:

- `rollup('A', 'B')` — hierarchical: produces `(A, B)` detail rows + `(A, NULL)` A-level subtotals + `(NULL, NULL)` grand total. **Does NOT produce `(NULL, B)` subtotals.**
- `cube('A', 'B')` — all combinations: produces `(A, B)`, `(A, NULL)`, `(NULL, B)`, and `(NULL, NULL)`. With n columns, generates 2ⁿ combinations.

Use `rollup` when hierarchy matters (country → state → city). Use `cube` when you need all cross-dimensional subtotals.
</details>

<details>
<summary>Q5: Why should you specify pivot values explicitly?</summary>

**A:** Without explicit values, Spark must perform **two full passes** over the data:
1. First pass: scan all data to discover unique pivot values
2. Second pass: perform the actual aggregation

With explicit values (e.g., `pivot('quarter', ['Q1', 'Q2', 'Q3', 'Q4'])`), Spark performs only **one pass**. On large datasets, this can cut execution time in half. Always provide pivot values when you know them in advance.
</details>